In [ ]:
from neo4j import GraphDatabase
from google import genai
from google.genai import types
import json
from langchain_core.documents import Document
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
import chromadb
import os

In [ ]:
URI = "bolt://localhost:7687"
USER = os.getenv("NEO_USER", "neo4j")
PASSWORD = os.getenv("NEO_PASSWORD", "password")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

Once again using an externally run LLM as the ones I was able to run locally were not only slow, but also not very good at understanding the context and rules.

In [ ]:
client_genai = genai.Client(api_key="MY_API_KEY")

This function normalizes the names of entities and relationships, as some Characters have different names in the books and series, different arrays are needed for each.

In [ ]:
aliases_book = {
    "Ali": "Alison DiLaurentis",
    "Alison": "Alison DiLaurentis",
    "Ezra": "Ezra Fitz",
    "Mr. Fitz": "Ezra Fitz",
    "Hanna": "Hanna Marin",
    "Emily": "Emily Fields",
    "Aria": "Aria Montgomery",
    "Spencer": "Spencer Hastings",
    "Maya": "Maya St. Germain",
    "Melissa": "Melissa Hastings",
    "Wren": "Wren Kim",
    "Jessica": "Jessica DiLaurentis",
    "Toby": "Toby Cavanaugh",
    "Jenna": "Jenna Marshall",
    "Wilden": "Darren Wilden",
    "Detective Wilden": "Darren Wilden",
    "Sean": "Sean Ackard",
    "Ben": "Ben Coogan",
    "Mona": "Mona Vanderwaal",
    "Meredith": "Meredith Succi",
    "Byron": "Byron Montgomery",
    "Ella": "Ella Montgomery",
    "Veronica": "Veronica Hastings",
    "Isabel": "Isabel Marin",
    "Amber": "Amber Victorino",
    "Peter": "Peter Hastings",
    "Andrew": "Andrew Newhoff",
    "Alex": "Alex Santiago",
    "Lucas": "Lucas Gottesman",
    "Ashley": "Ashley Marin",
    "Mike": "Mike Montgomery",
    "Toby Walker": "Toby Cavanaugh",
    "Toby Marshall": "Toby Cavanaugh",
    "Holden": "Holden Strauss",
    "The girls' clique": "The Liars",
    "The girls": "The Liars",
    "The Clique": "The Liars",
    "Pretty Little Liars": "The Liars",
    "The Liars (again)": "The Liars",
    "The Liars' Clique": "The Liars",
    "Paige": "Paige McCullers",
    "Paige McCullough": "Paige McCullers",
    "Officer Garrett Reynolds": "Garrett Reynolds",
    "Garrett": "Garrett Reynolds",
    "Dr. Anne Sullivan": "Dr. Sullivan",
    "Hanna Marin's father (Tom)": "Tom Marin",
    "the Marins": "The Marin Family",
    "the Fields' family": "The Fields Family",
    "Dr. Amanda Sullivan": "Dr. Sullivan",
    "Hanna Marin's father": "Tom Marin",
    "Holden": "Holden Straussberg",
    "Officer Wilden": "Darren Wilden", 
    "Wren Kingston": "Wren Kim",
    "Rylie": "Riley Wolfe",
    "Riley": "Riley Wolfe",
    "Trista": "Trista Taylor",
    "Ali (Alison)": "Alison DiLaurentis",
    "Ian": "Ian Thomas",
    "Noel": "Noel Kahn",
    "Andrew": "Andrew Campbell",
    "Andrew Newhoff": "Andrew Campbell",

    "FRIEND_OF": "FRIEND",
    "DATER_OF": "DATED",
    "DATING": "DATED",
    "LOVER_OF": "LOVES",
    "BLACKMAILED_BY": "BLACKMAILED",
    
}

aliases_series = {
    "Ali": "Alison DiLaurentis",
    "Alison": "Alison DiLaurentis",
    "Ezra": "Ezra Fitz",
    "Mr. Fitz": "Ezra Fitz",
    "Hanna": "Hanna Marin",
    "Emily": "Emily Fields",
    "Aria": "Aria Montgomery",
    "Spencer": "Spencer Hastings",
    "Maya": "Maya St. Germain",
    "Melissa": "Melissa Hastings",
    "Wren": "Wren Kingston",
    "Jessica": "Jessica DiLaurentis",
    "Toby": "Toby Cavanaugh",
    "Jenna": "Jenna Marshall",
    "Wilden": "Darren Wilden",
    "Detective Wilden": "Darren Wilden",
    "Sean": "Sean Ackard",
    "Ben": "Ben Coogan",
    "Mona": "Mona Vanderwaal",
    "Meredith": "Meredith Succi",
    "Byron": "Byron Montgomery",
    "Ella": "Ella Montgomery",
    "Veronica": "Veronica Hastings",
    "Isabel": "Isabel Marin",
    "Amber": "Amber Victorino",
    "Peter": "Peter Hastings",
    "Andrew": "Andrew Newhoff",
    "Alex": "Alex Santiago",
    "Lucas": "Lucas Gottesman",
    "Ashley": "Ashley Marin",
    "Mike": "Mike Montgomery",
    "Toby Walker": "Toby Cavanaugh",
    "Toby Marshall": "Toby Cavanaugh",
    "Holden": "Holden Strauss",
    "The girls' clique": "The Liars",
    "The girls": "The Liars",
    "The Clique": "The Liars",
    "Pretty Little Liars": "The Liars",
    "The Liars (again)": "The Liars",
    "The Liars' Clique": "The Liars",
    "Paige": "Paige McCullers",
    "Paige McCullough": "Paige McCullers",
    "Officer Garrett Reynolds": "Garrett Reynolds",
    "Garrett": "Garrett Reynolds",
    "Dr. Anne Sullivan": "Dr. Sullivan",
    "Hanna Marin's father (Tom)": "Tom Marin",
    "the Marins": "The Marin Family",
    "the Fields' family": "The Fields Family",
    "Dr. Amanda Sullivan": "Dr. Sullivan",
    "Hanna Marin's father": "Tom Marin",
    "Holden": "Holden Straussberg",
    "Officer Wilden": "Darren Wilden",
    "Officer Darren Wilden": "Darren Wilden",
    "The girls (Aria Montgomery, Spencer Hastings, Hanna Marin, Emily Fields)": "The Liars",
    "Dr. Wren Kingston": "Wren Kingston",
    "Caleb": "Caleb Rivers",
    "Alex Wolf": "Alex Santiago",
    "The Police": "The Rosewood Police Force",
    "Ian": "Ian Thomas",
    "Jackie": "Jackie Molina",
    "Darren": "Darren Wilden",
    "Noel": "Noel Kahn",
    "Andrew": "Andrew Campbell",
    "Andrew Newhoff": "Andrew Campbell",

    "FRIEND_OF": "FRIEND",
    "DATER_OF": "DATED",
    "DATING": "DATED",
    "LOVER_OF": "LOVES",
    "BLACKMAILED_BY": "BLACKMAILED", 
}

def normalize(name, type):
    if type == "book":
        return aliases_book.get(name, name)
    if type == "series":
        return aliases_series.get(name, name)

### 1. Predefining the functions for the graph database retrieval agent

##### Selecting a single node and all its relationships

In [38]:
def single_node(person):
    query_series = f"""
        MATCH (a {{name: "{normalize(person, "series")}"}})-[r:RELATED_TO]-(b)
        RETURN a.name AS source,
            b.name AS target,
            r.type AS rel_type,
            r.season AS season,
            r.episode AS episode
        ORDER BY r.season, r.episode
        LIMIT 50
"""
    query_books = f"""
        MATCH (a {{name: "{normalize(person, "book")}"}})-[r:RELATED_TO]-(b)
        RETURN a.name AS source,
            b.name AS target,
            r.type AS rel_type,
            r.book AS book
        ORDER BY r.book
        LIMIT 50
"""
    rows_books = "No nodes/relations found."
    with driver.session(database="books") as session:
        result = session.run(query_books)
        rows_books = [record.data() for record in result]

    rows_series = "No nodes/relations found."
    with driver.session(database="series") as session:
        result = session.run(query_series)
        rows_series = [record.data() for record in result]
    return rows_books, rows_series

##### Selecting all relationships between two nodes

In [39]:
def relationship_between(persona, personb):
    query_series = f"""
        MATCH (a {{name: "{normalize(persona, "series")}"}})-[r:RELATED_TO]-(b {{name: "{normalize(personb, "series")}"}})
        RETURN a.name AS source,
            b.name AS target,
            r.type AS rel_type,
            r.season AS season,
            r.episode AS episode
        ORDER BY r.season, r.episode
"""
    query_books = f"""
        MATCH (a {{name: "{normalize(persona, "book")}"}})-[r:RELATED_TO]-(b {{name: "{normalize(personb, "book")}"}})
        RETURN a.name AS source,
            b.name AS target,
            r.type AS rel_type,
            r.book AS book
        ORDER BY r.book
"""
    rows_books = "No relationships found."
    with driver.session(database="books") as session:
        result = session.run(query_books)
        if result.data != []:
            rows_books = [record.data() for record in result]

    rows_series = "No relationships found."
    with driver.session(database="series") as session:
        result = session.run(query_series)
        if result.data != []:
            rows_series = [record.data() for record in result]
    return rows_books, rows_series

##### Selecting all relationships and their nodes from one node, but only for a certain relationship type

In [40]:
def relationship(person, relationship_type):
    query_series=f"""
    MATCH (e {{name: "{normalize(person, "series")}"}})-[r:RELATED_TO]-(other)
    WHERE r.type = "{normalize(relationship_type, "series")}"
    RETURN e.name AS source,
        r.type AS rel_type,
        other.name AS target,
        r.season AS season,
        r.episode AS episode
    ORDER BY r.season, r.episode
"""
    
    query_books=f"""
    MATCH (e {{name: "{normalize(person, "book")}"}})-[r:RELATED_TO]-(other)
    WHERE r.type = "{normalize(relationship_type, "book")}"
    RETURN e.name AS source,
        r.type AS rel_type,
        other.name AS target,
        r.book as book
    ORDER BY r.book
"""
    rows_books = "No relationships found."
    with driver.session(database="books") as session:
        result = session.run(query_books)
        rows_books = [record.data() for record in result]

    rows_series = "No relationships found."
    with driver.session(database="series") as session:
        result = session.run(query_series)
        rows_series = [record.data() for record in result]
    return rows_books, rows_series

### 2. Defining the prompt and the return schema

In [ ]:
prompt = f"""
You are a routing model for a relationship entity graph in Neo4j.
Your task is NOT to answer the user's question yourself.

Your task is only to:
-understand the question,
-select the appropriate Python function,
-extract the correct arguments,
-and return the tool selection as JSON.

The graph contains characters and groups from a series/book.
Relationships between characters exist in Neo4j as:
(a)-[r:RELATED_TO]-(b)

Each relationship has one of these possible properties:
ENGAGED | FRIEND | FAMILY | SIBLING | STEP-SIBLING | PARENT | CHILD | DATED | MARRIED | BLACKMAILED | ALLIED

Please return one of the three dictionaries:

{{"function":"single_node", "person": person}}
    Purpose: Retrieves all relationships of a single character.

    Use this function when the question asks about one character
    and wants to know who they are connected to or what relationships they have.

    Examples:
    - "Who is Emily related to?"
    - "What relationships does Mona have?"
    - "Who are Emily's siblings?"
    - "What connections does Alison have in the series?"
    - "Who is Pam Fields' family?"

{{"function":"relationship_between", "persona": persona, "personb": personb}}
    Purpose: Retrieves all direct relationships between two specific characters.

    Use this function when the question explicitly names two characters
    and asks about their relationship to each other.

    Examples:
    - "What is the relationship between Emily and Alison?"
    - "What is going on between Hanna and Caleb?"
    - "Are Spencer and Toby together?"
    - "How are Mona and Hanna connected?"
    - "Is Emily dating Paige in the series?"

{{"function":"relationship", "persona": persona, "relationship_type": relationship}}
    Purpose: Retrieves only relationships of a specific type for a single character.

    Use this function when the question asks about a specific relationship type
    for exactly one character.

    Examples:
    - "Who is Hanna dating?"
    - "Who is Emily related to?"
    - "Who are Mona's friends?"
    - "Who are Alison's enemies?"
    - "How many siblings does Spencer have?"
    - "Does Aria have parents?"

Please return the appropriate Python function as a dictionary.

The question is:
"""

In [42]:
response_schema = {
    "type": "object",
    "oneOf": [
        {
            "properties": {
                "function": {"const": "single_node"},
                "person": {"type": "string"}
            },
            "required": ["function", "person"],
            "additionalProperties": False
        },
        {
            "properties": {
                "function": {"const": "relationship_between"},
                "persona": {"type": "string"},
                "personb": {"type": "string"}
            },
            "required": ["function", "persona", "personb"],
            "additionalProperties": False
        },
        {
            "properties": {
                "function": {"const": "relationship"},
                "persona": {"type": "string"},
                "relationship_type": {
                    "type": "string",
                    "enum": ["ENGAGED", "FRIEND", "FAMILY", "SIBLING", "STEP-SIBLING", "PARENT", "CHILD", "DATED", "MARRIED",  "BLACKMAILED", "ALLIED" ]
                }
            },
            "required": ["function", "persona", "relationship_type"],
            "additionalProperties": False
        }
    ]
}

### 3. Final Function for graph data retrieval

In [43]:
def get_graph_data(question):
    response = client_genai.models.generate_content(
        model="gemini-3.1-flash-lite",
        #model="gemma-4-26b-a4b-it",
        contents=prompt + question,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json",
        ),
)
    print(response.text)

    graph_context_books = None
    graph_context_series = None
    js = json.loads(response.text)

    if js["function"] == "single_node":
        graph_context_books, graph_context_series = single_node(js["person"])
    elif js["function"] == "relationship_between":
        graph_context_books, graph_context_series = relationship_between(js["persona"], js["personb"])
    elif js["function"] == "relationship":
        graph_context_books, graph_context_series = relationship(js["persona"], js["relationship_type"])
    
    return graph_context_books, graph_context_series

### 4. Vector Retrieval from vector database

In [45]:
client = chromadb.PersistentClient(path="chroma_db")
collection_books = client.get_collection(name="books")
collection_series = client.get_collection(name="series")

As explained in the report, instead of using a regular RAG system (which did not work too well in Exercise 3), I am using one with a reranker, which first selects a lot of suiting matches with a faster model and then uses a slower model to rerank those matches and use only the most matchin ones.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
model = SentenceTransformer("microsoft/harrier-oss-v1-270m")
reranker = CrossEncoder("Qwen/Qwen3-Reranker-0.6B")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2855.71it/s]


In [50]:
def retrieve_context(question, collection, top_n=20, top_k=5):
    q_emb = model.encode(question).tolist()

    series_res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_n,
        include=["documents", "metadatas", "distances"]
    )

    candidates = []
    for doc, meta, dist in zip(
        series_res["documents"][0],
        series_res["metadatas"][0],
        series_res["distances"][0]
    ):
        candidates.append({
            "text": doc,
            "meta": meta,
            "distance": dist
        })

    pairs = [[question, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    for c, score in zip(candidates, scores):
        c["rerank_score"] = float(score)

    candidates = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return candidates[:top_k]

### 5. Final RAG system with graph data retrieval

In [ ]:
def rag(ques):
    print("Generating graph context...")
    context_book_graph, context_series_graph = get_graph_data(ques)
    print("Book Graph")
    print(context_book_graph)
    print("Series Graph")
    print(context_series_graph)
    print("Generating vector context...")
    context_series_vector = retrieve_context(ques, collection_series)
    context_book_vector = retrieve_context(ques, collection_books)
    print("Book Vectors")
    print(context_book_vector)
    print("Series Vectors")
    print(context_series_vector)
    print("Caluclating an answer...")

    response = client_genai.models.generate_content(
        model="gemini-3.1-flash-lite",
        #model="gemma-4-26b-a4b-it",
        contents=f"""You are an expert analyst comparing the Pretty Little Liars book series to the TV show adaptation.

        ANSWER THIS QUESTION BASED ONLY ON THE PROVIDED CONTEXT below:
        QUESTION: {ques}

        PROVIDED CONTEXT:

        BOOKS - TEXT PASSAGES:
        {context_book_vector}

        BOOKS - GRAPH RELATIONSHIPS:
        {context_book_graph}

        SERIES - TEXT PASSAGES:
        {context_series_vector}

        SERIES - GRAPH RELATIONSHIPS:
        {context_series_graph}

        INSTRUCTIONS:
        Answer the question in a short, brief and direct answer, clearly state what happens in the BOOKS vs. what happens in the SERIES.
        Only use parts of the given context that are relevant to the question.
        If there is no fitting context or the context is empty assume that the characters or relationships do not exist.""",
        config=types.GenerateContentConfig(
            temperature=0.1
        ),)

    return response.text

### 6. Testing the system on each of the test questions

In [53]:
questions = ["Who is 'A'?", "Who is Caleb?", "Who did Spencer date?", "What happens to Ezra Fitz?", "What is the relationship between Melissa and Spencer?", "What is the relationship between Hanna and Lucas?"]

In [54]:
for question in questions:
    print(rag(question))

Generating graph context...
{"function": "single_node", "person": "A"}
Book Graph
[{'source': 'A', 'target': 'Aria Montgomery', 'rel_type': 'BLACKMAILED', 'book': 2}, {'source': 'A', 'target': 'Hanna Marin', 'rel_type': 'BLACKMAILED', 'book': 2}, {'source': 'A', 'target': 'Emily Fields', 'rel_type': 'BLACKMAILED', 'book': 2}]
Series Graph
[{'source': 'A', 'target': 'Aria Montgomery', 'rel_type': 'BLACKMAILED', 'season': 1, 'episode': 3}, {'source': 'A', 'target': 'Spencer Hastings', 'rel_type': 'BLACKMAILED', 'season': 1, 'episode': 3}, {'source': 'A', 'target': 'Hanna Marin', 'rel_type': 'BLACKMAILED', 'season': 1, 'episode': 3}, {'source': 'A', 'target': 'Emily Fields', 'rel_type': 'BLACKMAILED', 'season': 1, 'episode': 3}, {'source': 'A', 'target': 'Dr. Sullivan', 'rel_type': 'BLACKMAILED', 'season': 2, 'episode': 25}]
Generating vector context...
Book Vectors
[{'text': 'person who was a part of the incident besides the Liars. "A" sends Emily a text message hinting that "A" knows ab